In [1]:
import braceexpand
# from models.fmri_encoder import FMRI2CLIP
from models.fmri_c2f_encoder import FMRI2CLIP

/data1/zx/switti-right/models/basic_switti.py:25: UserWarning: Cannot import apex RMSNorm, switch to vanilla implementation
  warnings.warn("Cannot import apex RMSNorm, switch to vanilla implementation")


In [2]:
import os
import torch
import clip
import json
import re
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from transformers import LlamaForCausalLM, LlamaTokenizer
from torchvision import transforms
from tqdm import tqdm # Use standard tqdm for scripts
import numpy as np

DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"

SHIKRA_PATH = '/home/zhangx/.cache/huggingface/hub/shikra-7b'

ADAPTER_PATH = '/home/zhangx/.cache/huggingface/hub/shikra-7b/mm_projector.bin'

EXPR_PATH = 'bbox/coco_bbox_categorized.json'

from transformers import CLIPVisionModel, CLIPImageProcessor

# --- Step 2: Load All Necessary Models (Corrected) ---
print("Loading models...")
tokenizer = LlamaTokenizer.from_pretrained(SHIKRA_PATH, padding_side='left')
shikra_model = LlamaForCausalLM.from_pretrained(SHIKRA_PATH, torch_dtype=torch.float16).to(DEVICE)

# Define the correct vision shikra_model path that Shikra expects
VISION_TOWER_PATH = 'openai/clip-vit-large-patch14'

# Load the correct vision shikra_model and preprocessor from `transformers`
# This replaces `clip.load()`
print(f"Loading vision tower from {VISION_TOWER_PATH}...")
vision_encoder = CLIPVisionModel.from_pretrained(VISION_TOWER_PATH, torch_dtype=torch.float16).to(DEVICE)
clip_preprocess = CLIPImageProcessor.from_pretrained(VISION_TOWER_PATH)
vision_encoder.eval() # Set vision shikra_model to evaluation mode

# Now, the projector's input dimension (1024) correctly matches the vision_encoder's output.
mm_projector = torch.nn.Linear(1024, 4096)
mm_projector_weights = torch.load(ADAPTER_PATH, map_location='cpu')
adjusted_weights = {k.split('.')[-1]: v for k, v in mm_projector_weights.items()}
mm_projector.load_state_dict(adjusted_weights)
mm_projector.to(DEVICE).eval()
print("✅ Models loaded.")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You are using a model of type shikra to instantiate a model of type llama. This is not supported for all configurations of models and can yield errors.


Loading models...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading vision tower from openai/clip-vit-large-patch14...
✅ Models loaded.


In [3]:
subj = 1
if subj == 1:
    num_voxels = 15724
elif subj == 2:
    num_voxels = 14278
elif subj == 3:
    num_voxels = 15226
elif subj == 4:
    num_voxels = 13153
elif subj == 5:
    num_voxels = 13039
elif subj == 6:
    num_voxels = 17907
elif subj == 7:
    num_voxels = 12682
elif subj == 8:
    num_voxels = 14386

In [4]:
import torch
from models.helpers import DropPath
from models import Switti, VQVAE
from models.pipeline import SwittiPipeline
device = 'cuda:1'
pn = "1_2_3_4_6_9_13_18_24_32"
patch_nums = tuple(map(int, pn.replace("-", "_").split("_")))
switti = Switti(
            depth=30,
            embed_dim=30 * 64,
            num_heads=30,
            drop_rate=0,
            attn_drop_rate=0,
            drop_path_rate=0,
            norm_eps=1e-6,
            attn_l2_norm=True,
            patch_nums=patch_nums,
            rope=True,
            rope_theta=10000,
            rope_size=128,
            use_swiglu_ffn=True,
            use_ar=False,
            use_crop_cond=False,
        ).to(device)
vae_local = VQVAE(
    vocab_size=4096,
    z_channels=32,
    ch=160,
    test_mode=True,
    share_quant_resi=4,
    v_patch_nums=patch_nums,
).to(device)
# fmri_encoder = FMRI2CLIP().to(device)



# fmri_vitl = FMRI2CLIP(input_dim=num_voxels,
#                         d_model=768,
#                         fmri_seq_len=100,
#                         image_seq_len=257,
#                         text_seq_len=77,
#                         num_layers=10).to(device)

fmri_vitl = FMRI2CLIP(input_dim=num_voxels,
                    d_model=768,#768,
                    fmri_seq_len=100,
                    image_seq_len=257,
                    text_seq_len=77,
                    num_layers=4).to(device)

pipe = SwittiPipeline(switti, vae_local, fmri_vitl, device)


[constructor]  ==== fused_if_available=True (fusing_add_ln=0/30, fusing_mlp=0/30) ==== 
    [Switti config ] embed_dim=1920, num_heads=30, depth=30, mlp_ratio=4.0
    [drop ratios ] drop_rate=0, attn_drop_rate=0, drop_path_rate=0 (tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.]))



In [5]:
switti_path = f"local_output/subj{subj}_lr5e-7_1e-4_cos_vitL_hierachy_12to24_rightway_compare/model_state_dict.pt"
pipe.pretrained(switti_path, subj=subj, device=device, torch_dtype=torch.bfloat16, reso=512)

/home/zhangx/miniconda3/envs/mindeye2/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [6]:
import webdataset as wds
import h5py
import numpy as np
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
data_path = "/data1/zx/switti-right/dataset/mindeyev2/wds/oldval_test"
# prepare models and data loaders
print('prepare NSD webdataset data...')
val_url = f"{data_path}/webdataset_avg_split/test/test_subj0{subj}_" + "{0..1}.tar"
meta_url = f"{data_path}/webdataset_avg_split/metadata_subj0{subj}.json"
num_val = 982

prepare NSD webdataset data...


In [7]:
print('prepare train and validation dataloaders...')
to_tuple = ["voxels", "images"]
val_batch_size = 1
split_by_node = lambda urls: urls
val_url = list(braceexpand.braceexpand(val_url))
val_data = wds.WebDataset(val_url, resampled=False, nodesplitter=split_by_node) \
    .decode("torch")\
    .rename(images="jpg;png", voxels='nsdgeneral.npy', trial="trial.npy", coco="coco73k.npy", reps="num_uniques.npy") \
    .to_tuple(*to_tuple) \
    .batched(val_batch_size, partial=False)

val_dl = torch.utils.data.DataLoader(val_data, batch_size=None, num_workers=1, shuffle=False)

voxels_per_subj = {1: 15724, 2: 14278, 3: 15226, 4: 13153, 5: 13039, 6: 17907, 7: 12682, 8: 14386}
num_voxels = voxels_per_subj.get(subj)

prepare train and validation dataloaders...


In [8]:
# Assuming 'pipe', 'device', and 'val_dl' are already defined
import torch
from tqdm import tqdm
save_image=True
# This list will store the reconstructed image batches from the GPU
all_recons_list = []
print("Running inference on validation data...")
image_list = []
for voxel, image in tqdm(val_dl): 
    # Note: We use '_' to ignore the 'image' tensor from the dataloader
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float32):
        batch_voxels = torch.mean(voxel, axis=1).float().to(device)
        
        # 2. Run the inference pipe on the prepared batch
        batch_samples = pipe(
            prompt=batch_voxels, 
            cfg=6,
            top_k=400,
            top_p=0.95,
            return_pil=False,
            turn_on_cfg_start_si=1,
            seed=59
        )
        
        # 3. Move the results to the CPU to free up GPU memory
        all_recons_list.append(batch_samples.cpu())

        # 4. (Optional) Explicitly clean up memory to be safe
        del batch_voxels
        del batch_samples
        torch.cuda.empty_cache()
        if save_image:
            image_list.append(image) # for visualization

# --- Finalize Results ---
print("Concatenating final results...")
# Concatenate all the reconstructed batches into a single tensor
final_recons = torch.cat(all_recons_list, dim=0)

print(f"Final output shape: {final_recons.shape}")

Running inference on validation data...


982it [15:39,  1.05it/s]


Concatenating final results...
Final output shape: torch.Size([982, 3, 512, 512])


In [9]:
all_recons = final_recons

In [10]:
# import os
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# %env HF_ENDPOINT=https://hf-mirror.com
# %env HTTP_PROXY=http://127.0.0.1:7896
# %env HTTPS_PROXY=http://127.0.0.1:7896

# # Your code remains the same
# tokenizer = AutoTokenizer.from_pretrained(
#     "huggyllama/llama-7b",
# )
# shikra_model = AutoModelForCausalLM.from_pretrained(
#     "huggyllama/llama-7b",
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# print("shikra_model and tokenizer loaded successfully from hf-mirror.com!")

In [11]:
from utils.grounding_utils import postprocess, extract_boxes

In [12]:
imsize = 256
if all_recons.shape[-1] != imsize:
    all_recons = transforms.Resize((imsize,imsize))(all_recons).float()

/home/zhangx/miniconda3/envs/mindeye2/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


In [13]:
# from utils.grounding_utils import postprocess, extract_boxes
# import time
# from torchvision.transforms import ToPILImage
# # --- Step 3: Load Data and Define Prompt Structure ---
# print("Loading data and defining prompts...")
# system = "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER:"
# user_image = " <im_start>" + "<im_patch>" * 256 + "<im_end> "
# prompt = "Locate <expr> in <image> and provide its coordinates with four numbers, please."
# with open(EXPR_PATH, 'r') as f:
#     expr_dict = json.load(f)
# gen_kwargs = dict(
#     use_cache=True,
#     do_sample=False,
#     pad_token_id=2, # tokenizer.pad_token_id,
#     bos_token_id=1, # tokenizer.bos_token_id,
#     eos_token_id=2, # tokenizer.eos_token_id,
#     max_new_tokens=512,
# )
    
# rec_result = {}
# result_dir = f'rec_results/subj{subj}_right'
# os.makedirs(result_dir, exist_ok=True)
# for cur_image_idx in range(all_recons.shape[0]):
#     expr_list = list(expr_dict[str(cur_image_idx)].keys())
#     for expr in expr_list:
#         user_prompt = prompt.replace('<expr>', expr)
#         if '<image>' in user_prompt:
#             user_prompt = user_prompt.replace('<image>', user_image)
#             input_text = system + user_prompt + " ASSISTANT:"
#         else:
#             input_text = system + user_image + user_prompt + " ASSISTANT:"
#         print(input_text)
#         input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(DEVICE)
#         inputs_embeds = shikra_model.model.embed_tokens(input_ids)
        
#         with torch.no_grad():
#             pil_image = transforms.ToPILImage()(all_recons[cur_image_idx])
#             # 1. Use the new preprocessor from `transformers`
#             processed_images = clip_preprocess(images=pil_image, return_tensors="pt").to(DEVICE)
#             # 2. Get the raw patch embeddings (output has shape [batch_size, num_patches+1, 1024])
#             image_embeddings = vision_encoder(**processed_images).last_hidden_state.to(torch.float16)
#             # 3. Shikra uses the 256 patch tokens, not the [CLS] token, so we remove the first token
#             image_embeddings_patches = image_embeddings[:, 1:, :] # Shape: [batch_size, 256, 1024]
#             # 4. Project the 1024-dim features to 4096-dim. This will now work without error.
#             image_features = mm_projector(image_embeddings_patches.float()) # Shape: [batch_size, 256, 4096]
#             print(image_features.shape)
            
#         new_input_embeds = []
#         for cur_input_ids, cur_input_embeds in zip(input_ids, inputs_embeds):
#             cur_image_features = image_features[0]
#             num_patches = cur_image_features.shape[0]
#             image_start_tokens = torch.where(cur_input_ids == 32001)[0]
#             for image_start_token_pos in image_start_tokens:
#                 cur_image_features = image_features[0].to(device=cur_input_embeds.device)
#                 num_patches = cur_image_features.shape[0]
#                 if cur_input_ids[image_start_token_pos + num_patches + 1] != 32002:
#                     raise ValueError("The image end token should follow the image start token.")
                
#                 cur_new_input_embeds = torch.cat((cur_input_embeds[:image_start_token_pos + 1], cur_image_features,
#                                                         cur_input_embeds[image_start_token_pos + num_patches + 1:]), dim=0)
#             new_input_embeds.append(cur_new_input_embeds)
#         inputs_embeds = torch.stack(new_input_embeds, dim=0)
        
#         st_time = time.time()
#         with torch.inference_mode():
#             with torch.autocast(dtype=torch.float16, device_type='cuda'):
#                 output_ids = shikra_model.generate(inputs_embeds=inputs_embeds.float(), **gen_kwargs)
#         print(f"done generated in {time.time() - st_time} seconds")

#         # input_token_len = input_ids.shape[-1]
#         # input_text = tokenizer.batch_decode(input_ids)[0]
#         # response = tokenizer.batch_decode(output_ids[:, input_token_len:])[0]
#         response = tokenizer.batch_decode(output_ids)[0]

#         # print(f"input: {input_text}")
#         print(f"response for {expr} in image {cur_image_idx}: {response}")

#         # save response in a txt file
#         with open(os.path.join(result_dir, 'rec_response2.txt'), 'a') as f:
#             f.write(f'response_{expr}_{cur_image_idx}: ') # \n')
#             f.write(response + '\n')

#         # save result to a dict: {"0": {"umbrella": [[],[]], "carrot": [[]]}}
#         if str(cur_image_idx) not in rec_result:
#             rec_result[str(cur_image_idx)] = {}
#         rec_result[str(cur_image_idx)][expr] = extract_boxes(response)
#         # print(f"rec_result: {rec_result}")
#         # save processed image (only for bbox tasks)
#         save_image=False
#         if save_image:
#             _, processed_image = postprocess(response, image=ToPILImage()(image_list[cur_image_idx][0]), width=5)
#             # _, processed_image = postprocess(response, image=pil_image, width=5)
#             if processed_image is not None:
#                 output_path = os.path.join(result_dir, f'{cur_image_idx}_{expr}_prompt.png')
#                 processed_image.save(output_path)

In [17]:
from utils.grounding_utils import postprocess, extract_boxes
import time
from torchvision.transforms import ToPILImage
# --- Step 3: Load Data and Define Prompt Structure ---
print("Loading data and defining prompts...")
# --- Step 3: Load Data and Define Prompt Structure ---
print("Loading data and defining prompts...")
system = "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER:"
user_image = " <im_start>" + "<im_patch>" * 256 + "<im_end> "
prompt = "Locate <expr> in <image> and provide its coordinates with four numbers, please."

with open(EXPR_PATH, 'r') as f:
    expr_dict = json.load(f)

gen_kwargs = dict(
    use_cache=True,
    do_sample=False,
    pad_token_id=2, # tokenizer.pad_token_id,
    bos_token_id=1, # tokenizer.bos_token_id,
    eos_token_id=2, # tokenizer.eos_token_id,
    max_new_tokens=512,
)

rec_result = {}
# Assuming 'subj' is defined elsewhere
result_dir = f'rec_results/subj{subj}_right'
os.makedirs(result_dir, exist_ok=True)


# --- Timing Start ---
total_start_time = time.time()

for cur_image_idx in range(all_recons.shape[0]):
    expr_list = list(expr_dict.get(str(cur_image_idx), {}).keys())
    for expr in expr_list:
        user_prompt = prompt.replace('<expr>', expr)
        if '<image>' in user_prompt:
            user_prompt = user_prompt.replace('<image>', user_image)
            input_text = system + user_prompt + " ASSISTANT:"
        else:
            input_text = system + user_image + user_prompt + " ASSISTANT:"
        
        # print(input_text) # This print can be verbose and slow down the process
        input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(DEVICE)
        inputs_embeds = shikra_model.model.embed_tokens(input_ids)
        
        with torch.no_grad():
            pil_image = transforms.ToPILImage()(all_recons[cur_image_idx])
            processed_images = clip_preprocess(images=pil_image, return_tensors="pt").to(DEVICE)
            image_embeddings = vision_encoder(**processed_images).last_hidden_state.to(torch.float16)
            image_embeddings_patches = image_embeddings[:, 1:, :]
            image_features = mm_projector(image_embeddings_patches.float())
            # print(image_features.shape) # This print can be verbose

        new_input_embeds = []
        for cur_input_ids, cur_input_embeds in zip(input_ids, inputs_embeds):
            cur_image_features = image_features[0]
            num_patches = cur_image_features.shape[0]
            image_start_tokens = torch.where(cur_input_ids == 32001)[0]
            for image_start_token_pos in image_start_tokens:
                cur_image_features = image_features[0].to(device=cur_input_embeds.device)
                num_patches = cur_image_features.shape[0]
                if cur_input_ids[image_start_token_pos + num_patches + 1] != 32002:
                    raise ValueError("The image end token should follow the image start token.")
                
                cur_new_input_embeds = torch.cat((cur_input_embeds[:image_start_token_pos + 1], cur_image_features,
                                                  cur_input_embeds[image_start_token_pos + num_patches + 1:]), dim=0)
                new_input_embeds.append(cur_new_input_embeds)
        inputs_embeds = torch.stack(new_input_embeds, dim=0)
        
        st_time = time.time()
        with torch.inference_mode():
            with torch.autocast(dtype=torch.float16, device_type='cuda'):
                output_ids = shikra_model.generate(inputs_embeds=inputs_embeds.float(), **gen_kwargs)
        # print(f"done generated in {time.time() - st_time} seconds") # This measures only the generation step

        response = tokenizer.batch_decode(output_ids)[0]
        # print(f"Response for '{expr}' in image {cur_image_idx}: {response.strip()}")

        # save response in a txt file
        # with open(os.path.join(result_dir, 'rec_response2.txt'), 'a') as f:
        #     f.write(f'response_{expr}_{cur_image_idx}: ')
        #     f.write(response + '\n')

        # save result to a dict
        if str(cur_image_idx) not in rec_result:
            rec_result[str(cur_image_idx)] = {}
        rec_result[str(cur_image_idx)][expr] = extract_boxes(response)

# --- Timing End ---
total_end_time = time.time()
total_duration = total_end_time - total_start_time
print("-" * 50)
print(f"Total processing time: {total_duration:.2f} seconds")
print("-" * 50)

Loading data and defining prompts...
Loading data and defining prompts...
--------------------------------------------------
Total processing time: 1563.39 seconds
--------------------------------------------------


In [ ]:
with open(os.path.join(result_dir, 'rec_response2.json'), 'w') as f:
    json.dump(rec_result, f, indent=4)